## Visual Analyzer

Este notebook analiza visualmente los frames generados por el Notebook 01 - Asset Decomposer.

A diferencia de la primera versión, este notebook no depende de que exista una carpeta previa en `/content/output`.

Flujo:

1. Cargar `output.zip`.
2. Descomprimir el paquete.
3. Validar estructura:
   - `metadata.json`
   - `manifest.json`
   - `frames/`
4. Ejecutar OCR sobre los frames.
5. Generar análisis por frame.
6. Generar resumen visual global.

Entradas:

- `output.zip`

Salidas:

- `output/visual_analysis.json`
- `output/frame_analysis/*.json`

In [ ]:
#Instalación de requerientos

!pip install easyocr
!pip install opencv-python
!pip install pandas
!pip install pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 29.0 MB/s eta 0:00:00


# 1. Importación de librerías

In [ ]:
import os
import cv2
import json
import zipfile
import shutil
import easyocr
import numpy as np
import pandas as pd

from PIL import Image
from IPython.display import display
from google.colab import files

## 2. Cargar paquete generado por Notebook 01

Sube el archivo `output.zip` generado por el Asset Decomposer.

In [ ]:
uploaded = files.upload()

zip_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(".zip")
]

if not zip_files:
    raise ValueError("No se cargó ningún archivo .zip. Sube el output.zip generado por Notebook 01.")

ZIP_PATH = zip_files[0]

print(f"Archivo ZIP cargado: {ZIP_PATH}")

Saving output.zip to output.zip
Archivo ZIP cargado: output.zip


## 3. Limpiar entorno y descomprimir paquete

Esta celda evita errores por outputs anteriores en la sesión.

In [ ]:
OUTPUT_DIR = "/content/output"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall("/content")

print("ZIP descomprimido en /content")

ZIP descomprimido en /content


## 4. Validar estructura del paquete

El notebook espera encontrar:

```text
/content/output/
├── metadata.json
├── manifest.json
└── frames/

In [ ]:
MANIFEST_PATH = os.path.join(
    OUTPUT_DIR,
    "manifest.json"
)

METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "metadata.json"
)

FRAMES_DIR = os.path.join(
    OUTPUT_DIR,
    "frames"
)

required_paths = {
    "OUTPUT_DIR": OUTPUT_DIR,
    "MANIFEST_PATH": MANIFEST_PATH,
    "METADATA_PATH": METADATA_PATH,
    "FRAMES_DIR": FRAMES_DIR
}

for label, path in required_paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No se encontró {label}: {path}. Verifica que el ZIP venga del Notebook 01."
        )

print("Estructura validada correctamente.")

Estructura validada correctamente.


## 5. Explorar carpeta cargada

In [ ]:
for root, dirs, files_list in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 4 * (level + 1)

    for file in files_list[:10]:
        print(f"{subindent}{file}")

    if len(files_list) > 10:
        print(f"{subindent}... {len(files_list) - 10} archivos más")

output/
    manifest.json
    audio.wav
    metadata.json
    frames/
        frame_0008.jpg
        frame_0000.jpg
        frame_0006.jpg
        frame_0003.jpg
        frame_0002.jpg
        frame_0010.jpg
        frame_0007.jpg
        frame_0001.jpg
        frame_0009.jpg
        frame_0004.jpg
        ... 1 archivos más


## 6. Cargar metadata y manifest

In [ ]:
with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

print("Metadata:")
display(metadata)

print("Manifest:")
display({
    "duration": manifest.get("duration"),
    "audio_present": manifest.get("audio_present"),
    "frames_count": len(manifest.get("frames", []))
})

Metadata:


{'filename': 'OFERTA 1 ESCOLAR 6.01.26_V3.mp4',
 'duration': 10.01,
 'fps': 29.97002997002997,
 'width': 1920,
 'height': 1080,
 'orientation': 'horizontal',
 'audio_present': True}

Manifest:


{'duration': 10.01, 'audio_present': True, 'frames_count': 11}

## 7. Inicializar OCR

Se usa EasyOCR en español e inglés.

In [ ]:
reader = easyocr.Reader(
    ["es", "en"],
    gpu=False
)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

## 8. Preparar carpeta de análisis por frame

In [ ]:
FRAME_ANALYSIS_DIR = os.path.join(
    OUTPUT_DIR,
    "frame_analysis"
)

os.makedirs(
    FRAME_ANALYSIS_DIR,
    exist_ok=True
)

print(f"Carpeta creada: {FRAME_ANALYSIS_DIR}")

Carpeta creada: /content/output/frame_analysis


## 9. Procesar frames con OCR

Por cada frame:

- Detecta texto.
- Cuenta palabras.
- Guarda bounding boxes.
- Genera un JSON individual.

In [ ]:
all_frames_analysis = []

frames = manifest.get("frames", [])

if not frames:
    raise ValueError("El manifest no contiene frames.")

for frame_info in frames:

    frame_file = frame_info["file"]

    frame_path = os.path.join(
        FRAMES_DIR,
        frame_file
    )

    if not os.path.exists(frame_path):
        print(f"Frame no encontrado, se omite: {frame_path}")
        continue

    result = reader.readtext(frame_path)

    frame_words = 0
    frame_boxes = []
    frame_text = []

    for detection in result:

        bbox = detection[0]
        text = detection[1]
        confidence = float(detection[2])

        # Convert numpy floats in bbox to standard Python floats
        processed_bbox = []
        for point in bbox:
            processed_bbox.append([float(coord) for coord in point])

        words = len(text.split())
        frame_words += words
        frame_text.append(text)

        frame_boxes.append(
            {
                "text": text,
                "confidence": confidence,
                "bbox": processed_bbox # Use the processed bbox
            }
        )

    frame_analysis = {
        "frame": frame_file,
        "time": frame_info.get("time"),
        "text_detected": len(frame_text) > 0,
        "word_count": frame_words,
        "text": frame_text,
        "bounding_boxes": frame_boxes
    }

    all_frames_analysis.append(frame_analysis)

    output_file = os.path.join(
        FRAME_ANALYSIS_DIR,
        frame_file.replace(".jpg", ".json")
    )

    with open(output_file, "w") as f:
        json.dump(frame_analysis, f, indent=4)

print(f"{len(all_frames_analysis)} frames procesados.")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume

11 frames procesados.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 9.1. Extraer colores dominantes

k-means (OpenCV) sobre una muestra de pixeles de todos los frames extraídos.
Sirve para comparar contra `brief.brand.brand_colors_hex` en el motor de
reglas (`VISUAL_DOMINANT_COLOR`). No requiere instalar nada nuevo: `cv2` y
`numpy` ya se usan/importan en este notebook.

In [ ]:
DOMINANT_COLOR_K = 5
MAX_SAMPLES_PER_FRAME = 2000

cv2.setRNGSeed(42)  # reproducibilidad entre corridas del mismo asset

all_pixel_samples = []

for frame_info in frames:

    frame_file = frame_info["file"]

    frame_path = os.path.join(
        FRAMES_DIR,
        frame_file
    )

    if not os.path.exists(frame_path):
        continue

    bgr = cv2.imread(frame_path)
    if bgr is None:
        continue

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    pixels = rgb.reshape(-1, 3)

    if len(pixels) > MAX_SAMPLES_PER_FRAME:
        idx = np.random.choice(len(pixels), MAX_SAMPLES_PER_FRAME, replace=False)
        pixels = pixels[idx]

    all_pixel_samples.append(pixels)

if all_pixel_samples:

    pixel_pool = np.vstack(all_pixel_samples).astype(np.float32)
    k = min(DOMINANT_COLOR_K, len(pixel_pool))

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 0.5)
    _, labels, centers = cv2.kmeans(
        pixel_pool, k, None, criteria, attempts=3, flags=cv2.KMEANS_RANDOM_CENTERS
    )

    cluster_sizes = np.bincount(labels.flatten(), minlength=k)
    order = np.argsort(-cluster_sizes)  # más grande (más prevalente) primero

    dominant_colors = [
        "#{:02X}{:02X}{:02X}".format(*[int(round(c)) for c in centers[i]])
        for i in order
    ]

else:
    dominant_colors = []

print(f"Colores dominantes ({len(dominant_colors)}): {dominant_colors}")

## 10. Generar resumen visual global

In [ ]:
frames_processed = len(all_frames_analysis)

frames_with_text = sum(
    1
    for frame in all_frames_analysis
    if frame["text_detected"]
)

total_words = sum(
    frame["word_count"]
    for frame in all_frames_analysis
)

avg_words = (
    total_words / frames_processed
    if frames_processed > 0
    else 0
)

max_words = max(
    [frame["word_count"] for frame in all_frames_analysis],
    default=0
)

visual_analysis = {
    "asset_filename": metadata.get("filename"),
    "frames_processed": frames_processed,
    "frames_with_text": frames_with_text,
    "frames_without_text": frames_processed - frames_with_text,
    "total_words": total_words,
    "avg_words": round(avg_words, 2),
    "max_words_in_frame": max_words,
    "dominant_colors": dominant_colors
}

visual_analysis

{'asset_filename': 'OFERTA 1 ESCOLAR 6.01.26_V3.mp4',
 'frames_processed': 11,
 'frames_with_text': 11,
 'frames_without_text': 0,
 'total_words': 472,
 'avg_words': 42.91,
 'max_words_in_frame': 82}

## 11. Guardar visual_analysis.json